# 🚄 Notebook 4: Dual-Ended Differential Doppler Kinematics on Air Track

Welcome to the **Differential Acoustic Doppler Laboratory** (`v1.2.0`).

This notebook demonstrates precision 1D motion tracking of an air track glider using counter-positioned microphones at opposite ends of the track ($x=0$ and $x=L$):
* **Simultaneous Dual-ADC Sampling:** True zero inter-channel skew ($0.00\,\mu\text{s}$) across MAX4466 microphones on pins A0 and A1.
* **Common-Mode Thermal & Battery Drift Cancellation:**
  $$f_0(t) = \frac{f_1(t) + f_2(t)}{2}$$
  Both microphones observe the active buzzer simultaneously, completely canceling oscillator frequency drift!
* **Exact Differential Velocity Inversion (Double Sensitivity):**
  $$v(t) = c(T) \cdot \left(\frac{f_2(t) - f_1(t)}{f_1(t) + f_2(t)}ight)$$
* **Pneumatic Blower Squelch Gating:** Narrowband adaptive tracking band rejects air track hiss.
* **Aerodynamic Kinematics Extraction:** Measures air cushion viscous damping $\gamma$ ($v(t) = v_0 e^{-\gamma t}$) and bumper coefficient of restitution $e$.


## 1. Multi-Second Glider Flight Recording & Kinematic Trajectory

Place the active buzzer on the air track glider. Position **Mic 1 (A0)** at the left bumper ($x = 0$) and **Mic 2 (A1)** at the right bumper ($x = L$).

Run the cell below, release/push the glider when prompted, and observe the extracted velocity, acceleration, and position trajectories.


In [ ]:
import json
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

from pynq_localizer import MicrophoneArrayOverlay, KinematicAnalytics

print("=" * 76)
print("🚄 STEP 2.6: DUAL-ENDED DIFFERENTIAL DOPPLER AIR TRACK RUN")
print("=" * 76)

ol = None
try:
    print("⏳ [1/3] Initializing FPGA Hardware Overlay (Profile: 'audio', 50 kSPS)...")
    ol = MicrophoneArrayOverlay()
    
    # 1. Flight Parameters
    duration_flight_sec = 5.0
    track_len_m = 2.0  # 2.0 meter air track
    profile_path = Path("profiles/active_buzzer_2610hz.json")

    print(f"      • Track Length L     : {track_len_m:.2f} m")
    print(f"      • Flight Duration    : {duration_flight_sec:.1f} s")
    print(f"      • Sample Rate        : {ol.fs_per_ch:.0f} SPS per channel")

    print("\n" + "-" * 76)
    print("📍 [2/3] GLIDER FLIGHT RECORDING PROTOCOL")
    print("   1. Turn the buzzer ON (wire in 3.3V via 2N2222A).")
    print("   2. Turn ON the air track blower.")
    print("   3. Hold the glider at one end.")
    print("   4. Press [Enter] and IMMEDIATELY PUSH the glider across the track!")
    print("-" * 76)
    input("👉 Press [Enter] and launch the glider...")

    # Record and process continuous 100 Hz kinematic trajectory
    flight = ol.record_differential_flight(
        duration_sec=duration_flight_sec,
        track_length_m=track_len_m,
        profile=profile_path,
        temperature_c=20.0,
        window_ms=40.0,
        hop_ms=10.0
    )

    t_sec = flight["times_sec"]
    v_mps = flight["velocity_mps"]
    v_cmps = flight["velocity_cmps"]
    pos_m = flight["position_m"]
    f1 = flight["f_mic1_hz"]
    f2 = flight["f_mic2_hz"]
    f0_drift = flight["f0_common_hz"]
    summary = flight["kinematics_summary"]

    print("\n" + "=" * 76)
    print("📊 GLIDER FLIGHT KINEMATICS SUMMARY")
    print("=" * 76)
    print(f"  • Trajectory Points Extracted   : {len(t_sec)} frames (100 Hz rate)")
    print(f"  • Maximum Forward Speed (+v)    : {summary['max_forward_velocity_mps']*100:.1f} cm/s ({summary['max_forward_velocity_mps']:.3f} m/s)")
    print(f"  • Maximum Reverse Speed (-v)    : {summary['max_reverse_velocity_mps']*100:.1f} cm/s ({summary['max_reverse_velocity_mps']:.3f} m/s)")
    print(f"  • Total Bumper Bounces Detected : {summary['total_collisions_detected']}")
    if np.isfinite(summary.get("mean_coefficient_of_restitution", np.nan)):
        print(f"  • Bumper Restitution Coeff (e)  : e = {summary['mean_coefficient_of_restitution']:.3f} (Elasticity)")
    print(f"  • Air Viscous Damping Coeff (γ) : γ = {summary['viscous_drag_gamma']:.4f} s⁻¹")
    print(f"  • Total Glider Travel Distance  : {pos_m[-1] - pos_m[0]:.3f} m")
    print("=" * 76)

    # Render 3-Panel Interactive Kinematic Trajectory Dashboard
    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=(
            "<b>1. Dual Observed Frequencies (f1 Blue vs f2 Red) & Common Drift f0(t) [Hz]</b>",
            f"<b>2. Instantaneous Differential Velocity v(t) [cm/s] (Max = {np.max(np.abs(v_cmps)):.1f} cm/s)</b>",
            "<b>3. Integrated Glider Position Along Track x(t) [m]</b>"
        )
    )

    # Panel 1: Frequencies
    fig.add_scatter(x=t_sec, y=f1, mode="lines", line=dict(color="#00FFCC", width=1.8), name="Mic 1 (Left)", row=1, col=1)
    fig.add_scatter(x=t_sec, y=f2, mode="lines", line=dict(color="#FF007F", width=1.8), name="Mic 2 (Right)", row=1, col=1)
    fig.add_scatter(x=t_sec, y=f0_drift, mode="lines", line=dict(color="#FFA500", width=1.5, dash="dash"), name="Drift f0(t)", row=1, col=1)

    # Panel 2: Velocity
    fig.add_scatter(x=t_sec, y=v_cmps, mode="lines+markers", marker=dict(size=3), line=dict(color="#FFD600", width=2.0), name="v_diff (cm/s)", row=2, col=1)
    fig.add_hline(y=0.0, line=dict(color="gray", dash="dash"), row=2, col=1)

    # Panel 3: Position
    fig.add_scatter(x=t_sec, y=pos_m, mode="lines", line=dict(color="#76FF03", width=2.2), name="Position x(t)", row=3, col=1)
    fig.add_hline(y=0.0, line=dict(color="white", dash="dot"), annotation_text="Left Bumper (0m)", row=3, col=1)
    fig.add_hline(y=track_len_m, line=dict(color="white", dash="dot"), annotation_text=f"Right Bumper ({track_len_m}m)", row=3, col=1)

    fig.update_layout(template="plotly_dark", height=800, margin=dict(l=55, r=25, t=50, b=30))
    fig.update_yaxes(title="Frequency (Hz)", row=1, col=1)
    fig.update_yaxes(title="Velocity (cm/s)", row=2, col=1)
    fig.update_yaxes(title="Position (m)", range=[-0.05, track_len_m + 0.05], row=3, col=1)
    fig.update_xaxes(title="Time (Seconds)", row=3, col=1)
    fig.show()

finally:
    if ol is not None:
        ol.close()
        print("🔒 FPGA hardware resources cleanly released.")


## 2. Launch the Real-Time 4-Tab Kinematics & Doppler Dashboard

Run the cell below to launch the **100 Hz live streaming dashboard**. 
Click **Tab 3 (🔀 Dual Overlay & Doppler)** to observe real-time differential velocity $v_{\text{diff}}(t)$ in Row 4 while manually pushing the glider back and forth!


In [ ]:
from pynq_localizer import MicrophoneArrayOverlay

ol = MicrophoneArrayOverlay()

# Launch the live 4-tab instrument with AoA and Differential Doppler enabled
app = ol.kinematics_dashboard(
    window_duration_sec=10.0,
    hop_ms=10.0,
    aoa_mic_distance_m=0.05
)
